In [11]:
import numpy as np
import matplotlib.pyplot as plt


In [12]:
espacio_de_Fock = []

for n0 in range(7):
    for n1 in range(7):
        for n2 in range(7):
            for n3 in range(7):
                for n4 in range(7):
                    for n5 in range(7):
                        estado = np.array([n0, n1, n2, n3, n4, n5])
                        if np.sum(estado) == 6:
                            espacio_de_Fock.append(estado)

espacio_de_Fock = np.array(espacio_de_Fock)


print("Espacio de Fock:")
print(espacio_de_Fock[:10])  # Mostrar solo los primeros 10 estados para no saturar la salida

Espacio de Fock:
[[0 0 0 0 0 6]
 [0 0 0 0 1 5]
 [0 0 0 0 2 4]
 [0 0 0 0 3 3]
 [0 0 0 0 4 2]
 [0 0 0 0 5 1]
 [0 0 0 0 6 0]
 [0 0 0 1 0 5]
 [0 0 0 1 1 4]
 [0 0 0 1 2 3]]


In [13]:
def E_U(estado):
    return np.sum(estado * (estado - 1))


energias_U = [E_U(estado) for estado in espacio_de_Fock]
H_U = np.diag(energias_U)

In [14]:
dimension = len(espacio_de_Fock)
T = np.zeros((dimension, dimension), dtype=complex)
indice_estado = {tuple(estado): i for i, estado in enumerate(espacio_de_Fock)}

for indice_inicial, estado in enumerate(espacio_de_Fock):
    for x in range(6):
        y = (x + 1) % 6

        if estado[x] > 0:
            nuevo_estado = estado.copy()
            nuevo_estado[x] -= 1
            nuevo_estado[y] += 1

            A = np.sqrt(estado[x] * (estado[y] + 1))
            indice_nuevo = indice_estado[tuple(nuevo_estado)]
            T[indice_nuevo, indice_inicial] += A

K = T + T.conj().T

print(T)
print(K)

[[0.        +0.j 2.44948974+0.j 0.        +0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]
 [0.        +0.j 0.        +0.j 3.16227766+0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]
 [0.        +0.j 0.        +0.j 0.        +0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]
 ...
 [0.        +0.j 0.        +0.j 0.        +0.j ... 0.        +0.j
  1.        +0.j 0.        +0.j]
 [0.        +0.j 0.        +0.j 0.        +0.j ... 0.        +0.j
  0.        +0.j 2.44948974+0.j]
 [0.        +0.j 0.        +0.j 0.        +0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]]
[[0.        +0.j 2.44948974+0.j 0.        +0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]
 [2.44948974+0.j 0.        +0.j 3.16227766+0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]
 [0.        +0.j 3.16227766+0.j 0.        +0.j ... 0.        +0.j
  0.        +0.j 0.        +0.j]
 ...
 [0.        +0.j 0.        +0.j 0.        +0.j ... 0.        +0.j
  1.        +0.j 0.        +0.j]

In [ ]:
from scipy.linalg import expm


def H(t, kappa, omega):
    return H_U - kappa * omega * np.cos(omega * t) * K


def operador_evolucion(omega, kappa, M=40):
    if omega <= 0:
        raise ValueError("omega debe ser mayor que cero")

    periodo = 2 * np.pi / omega
    dt = periodo / M

    U = np.eye(dimension, dtype=complex)

    # Se utiliza el punto medio de cada intervalo temporal.
    tiempos = (np.arange(M) + 0.5) * dt

    for t in tiempos:
        H_t = H(t, kappa, omega)
        U_paso = expm(-1j * H_t * dt)

        # El paso temporal más reciente se multiplica por la izquierda.
        U = U_paso @ U

    return U


# w es un array; omega es una frecuencia concreta de ese array.
# Se empieza en 0.01 porque el periodo 2*pi/omega no existe para omega = 0.
w = np.linspace(0.01, 250, 8)
kappas = np.linspace(0, 1, 5)

for kappa in kappas:
    for omega in w:
        U_final = operador_evolucion(omega, kappa)

        # Analizar U_final y guardar solo el resultado que necesites.

KeyboardInterrupt: 